# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI, by de-duplicating from Andersen, and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import shutil 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 3): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")

# Dates and locations
start_date = "11-01-2021"
end_date = "07-25-2025"
date_range = start_date + "--" + end_date
locations = "Antarctica,North America,South America"

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" # "_Antarctica_North_America_South_America/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# andersen_dedup = home + "Andersen/complete/11-01-2021--06-13-2025/" # Use the previous date range for de-duplication!
# andersen_dedup2 = home + "Andersen/complete/06-14-2025--07-04-2025/"
andersen = home + "Andersen/complete/" + date_range + "/" # 11-01-2021--07-04-2025/"
# andersen_dedup = andersen
combined_files = home + "Combinations/Andersen_NCBI_Virus/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(combined_files): # checking if the directory exists or not
    os.makedirs(combined_files) # if the directory is not present then create it

# references = "C:/Users/maksi/Documents/Statistics/projects/Avian_Flu/references/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])

genotypes = ["B3.13", "D1.1", "D1.3"]



## Downloading Data

In [3]:


# browser = "Chrome"
# sleep_time = "3"
# locations = "South America"
# start_date = "07-01-2025"
# end_date = "07-25-2025"



In [4]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Get files
            open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 

## De-Duplication

In [5]:
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.SRA_Accession, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="SRA_Accession")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="first") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
# metadata_segments

111475


In [6]:
# De-duplicate from Andersen using SRA Accession

# If even one SRA Accession in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
andersen_sras = []
# Grab files
for dirpath, dirs, files in os.walk(andersen):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            for value in sra_accessions.values:
                if "SRR" in value:
                    andersen_sras.append(value)
            isolates = fasta_file["Isolate_Id"]
            for value in isolates.values:
                andersen_sras.append(value)
    break 

# Grab more files
# for dirpath, dirs, files in os.walk(andersen_dedup):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
#             sra_accessions = fasta_file["Identifier"]
#             for value in sra_accessions.values:
#                 if "SRR" in value:
#                     andersen_sras.append(value)
            
# Remove duplicates from Andersen
for value in andersen_sras: # to remove
    if "SRR" in value:
        metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    else:
        metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]
    # print(value)
    
print(len(metadata_segments))
# metadata_segments
# print(count)

2096


## Add sequences to dataframe

In [ ]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title

# Get sequences and headers together
headers = []
isolates = []
sras = []
headers_seqs = {}

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
# Extract segment number so that we can add the correct sequences to the correct sample
sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-1]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["SRA_Accession", "Segment"])

2096
2096


In [8]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PV949116.1,GenBank,GCA_051519285.1,SRR34270367,SAMN49684927,PRJNA1230736,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-11-25,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: NY|24-035330-001-orig...,ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCGCAGTCTCGCA...
1,PV949117.1,GenBank,GCA_051519285.1,SRR34270367,SAMN49684927,PRJNA1230736,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-11-25,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: NY|24-035330-001-orig...,ATGGATGTCAATCCGACTTTACTTTTCTTGAAAGTTCCAGCGCAAA...
2,PV949118.1,GenBank,GCA_051519285.1,SRR34270367,SAMN49684927,PRJNA1230736,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-11-25,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: NY|24-035330-001-orig...,ATGGAAGATTTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...
3,PV949119.1,GenBank,GCA_051519285.1,SRR34270367,SAMN49684927,PRJNA1230736,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-11-25,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: NY|24-035330-001-orig...,ATGGAAAGAATAGTGATTGCCCTCGCAATAATCAGCATTGTCAAAG...
4,PV949120.1,GenBank,GCA_051519285.1,SRR34270367,SAMN49684927,PRJNA1230736,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2024-11-25,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: NY|24-035330-001-orig...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2091,OQ565628.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,GTCAAAATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTG...
2092,OQ565629.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,TAGATAATCACTCACTGAGTGRYATSCACATCATGGCRTMYCARGR...
2093,OQ565630.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,CCATTGGATCARTCTGTATGGTAATTGGGATAGTCAGYTTGATGCT...
2094,OQ565631.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,TTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCT...


## Create FASTA files using deduplicated sequences

In [19]:
# Create 1 fasta file per header
metadata_segments["Partial_Header"] = metadata_segments["full_header"].apply(lambda x: "|".join(x.split("|")[:-1]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments[metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
            df["full_header"] = df["full_header"].apply(lambda x: x.replace(c, "_"))
            df_list.append(df)

# Make fasta files
for df in df_list:
    # Forbidden characters in file name 
    title = df["SRA_Accession"].values[0]
    # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    #     title = title.replace(c, "_")
    df_to_fasta(df, title + ".fasta", temp_files)

## Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND GENOTYPES.** Then, make sure "output.tsv" is in the downloads directory.

In [29]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("output.tsv", delimiter="\t")

metadata_segments["File Name"] = metadata_segments["SRA_Accession"] + ".fasta"

metadata_genoflu = metadata_segments.merge(output_genoflu, how="inner", on="File Name") 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill()

print(metadata_genoflu)


       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0     PV949116.1        GenBank  GCA_051519285.1   SRR34270367  SAMN49684927   
1     PV949117.1        GenBank  GCA_051519285.1   SRR34270367  SAMN49684927   
2     PV949118.1        GenBank  GCA_051519285.1   SRR34270367  SAMN49684927   
3     PV949119.1        GenBank  GCA_051519285.1   SRR34270367  SAMN49684927   
4     PV949120.1        GenBank  GCA_051519285.1   SRR34270367  SAMN49684927   
...          ...            ...              ...           ...           ...   
2091  OQ565628.1        GenBank  GCA_039342815.1   SRR23852495  SAMN33745292   
2092  OQ565629.1        GenBank  GCA_039342815.1   SRR23852495  SAMN33745292   
2093  OQ565630.1        GenBank  GCA_039342815.1   SRR23852495  SAMN33745292   
2094  OQ565631.1        GenBank  GCA_039342815.1   SRR23852495  SAMN33745292   
2095  OQ565632.1        GenBank  GCA_039342815.1   SRR23852495  SAMN33745292   

        BioProject      Organism_Name  

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [30]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['bubo virginianus', 'phasianinae', 'meleagris gallopavo', 'parabuteo unicinctus', 'corvus brachyrhynchos', 'anser caerulescens', 'anas carolinensis', 'haliaeetus leucocephalus', 'sus scrofa', 'columbidae', 'corvus corax', 'buteo jamaicensis', 'bos taurus', 'passer domesticus', 'anas platyrhynchos', 'mareca americana', 'calidris mauri', 'branta canadensis', 'capra hircus', 'mephitidae', 'corvus', 'felis catus', 'cygnus olor', 'accipiter cooperii', 'gallus gallus', 'mus musculus', 'anatidae', 'puma concolor', 'larus occidentalis', 'numididae', 'mareca strepera', 'pavo', 'cathartes aura', 'dromaius novaehollandiae', 'aix sponsa', 'pelecanus']
[]
           wild_avian domestic_avian               cattle        feline  \
0    great_horned_owl       pheasant            dairy_cow           cat   
1        common_raven         turkey               cattle  domestic_cat   
2       cooper's_hawk        chicken  cattle milk product     feral_cat   
3        coopers_hawk          goose          bo

In [41]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        "USA"
                                                                        )

# Make new labels
names = ">" + metadata_genoflu["SRA_Accession"].apply(lambda x: x.split(",")[0]) + "|A/" + metadata_genoflu["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata_genoflu["Geo_Location"] + "/" + metadata_genoflu["Isolate"] + "/" + metadata_genoflu["Years"].apply(lambda x: str(x)) + "|" + metadata_genoflu["Genotype_x"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype_y"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Geo_Location_Abrv"])


0       USA-NY
1       USA-NY
2       USA-NY
3       USA-NY
4       USA-NY
         ...  
2091       USA
2092       USA
2093       USA
2094       USA
2095       USA
Name: Geo_Location_Abrv, Length: 2096, dtype: object


## Rename segments and make complete FASTA files

In [42]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype_y"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype_y"] == genotype)] 
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

B3.13_PB2
D1.1_PB2
D1.3_PB2
B3.13_PB1
D1.1_PB1
D1.3_PB1
B3.13_PA
D1.1_PA
D1.3_PA
B3.13_HA
D1.1_HA
D1.3_HA
B3.13_NP
D1.1_NP
D1.3_NP
B3.13_NA
D1.1_NA
D1.3_NA
B3.13_MP
D1.1_MP
D1.3_MP
B3.13_NS
D1.1_NS
D1.3_NS


In [43]:
print(metadata_genoflu[metadata_genoflu["Genotype_y"] == "D1.1"])

       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
176   PV792578.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
177   PV792579.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
178   PV792580.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
179   PV792581.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
180   PV792582.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
...          ...            ...              ...           ...           ...   
1739  PQ798073.1        GenBank  GCA_046436945.1   SRR31347620  SAMN44727531   
1740  PQ798074.1        GenBank  GCA_046436945.1   SRR31347620  SAMN44727531   
1741  PQ798075.1        GenBank  GCA_046436945.1   SRR31347620  SAMN44727531   
1742  PQ798076.1        GenBank  GCA_046436945.1   SRR31347620  SAMN44727531   
1743  PQ798077.1        GenBank  GCA_046436945.1   SRR31347620  SAMN44727531   

       BioProject      Organism_Name   

In [44]:
# Create FASTA files

os.chdir(complete_files)

names = []

for df in segment_genotype_dfs:
    print(df)
    if len(df["Genotype_y"].values[0]) > 0:
        # print(df)
        file_name = df["Genotype_y"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        output_file.close()

       Accession GenBank_RefSeq         Assembly  \
128   PV904410.1        GenBank  GCA_051276625.1   
144   PV846570.1        GenBank  GCA_051134765.1   
160   PV792234.1        GenBank  GCA_050966735.1   
192   PV709410.1        GenBank  GCA_050792425.1   
200   PV709426.1        GenBank  GCA_050793975.1   
216   PV649192.1        GenBank  GCA_050512595.1   
224   PV649344.1        GenBank  GCA_050512855.1   
232   PV649424.1        GenBank  GCA_050512955.1   
256   PV457221.1        GenBank  GCA_049374065.1   
608   PV218574.1        GenBank  GCA_048432755.1   
616   PV218678.1        GenBank  GCA_048433455.1   
832   PV071666.1        GenBank  GCA_047521545.1   
840   PV071858.1        GenBank  GCA_047521985.1   
1360  PQ829074.1        GenBank  GCA_046510025.1   
1368  PQ829082.1        GenBank  GCA_046510665.1   
1376  PQ829114.1        GenBank  GCA_046510685.1   
1384  PQ829130.1        GenBank  GCA_046510705.1   
1392  PQ831034.1        GenBank  GCA_046511075.1   
1400  PQ8328

In [45]:
# Concatenate with new Andersen sequences

os.chdir(combined_files)

# andersen = home + "Andersen/complete/" + date_range + "/"

# Andersen files
filenames_andersen = []
for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
    for dirpath, dirs, files in os.walk(andersen): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            filenames_andersen.append(file_name)
        break 

# NCBI Virus files
filenames_ncbi = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

print(filenames_ncbi)

common_genotypes = set()
# Concatenate the two -- should not have any overlap due to dates and deduplication 
for a_file in filenames_andersen:
    partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
    for nv_file in filenames_ncbi:
        partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
        if partial_filename_a == partial_filename_nv:
            common_genotypes.add(partial_filename_a)
            filenames = [a_file, nv_file]
            with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
                        infile.close()
                outfile.close()

print(common_genotypes)

for a_file in filenames_andersen:
    partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
    # If genotype not found in NCBI Virus, include it as well
    if partial_filename_a not in common_genotypes:
        with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile2:
            with open(a_file) as infile2:
                for line in infile2:
                    outfile2.write(line)
                infile2.close()
            outfile2.close()

['C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/11-01-2021--07-25-2025_Antarctica_North_America_South_America/B3.13_HA_11-01-2021--07-25-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/11-01-2021--07-25-2025_Antarctica_North_America_South_America/B3.13_MP_11-01-2021--07-25-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/11-01-2021--07-25-2025_Antarctica_North_America_South_America/B3.13_NA_11-01-2021--07-25-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/11-01-2021--07-25-2025_Antarctica_North_America_South_America/B3.13_NP_11-01-2021--07-25-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/11-01-2021--07-25-2025_Antarctica_North_America_South_America/B3.13_NS_11-01-2021--07-25-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/11-01-2021--07-25-2025_Antarctica_North